# 15 · Streaming

**Goal:** by the end you can watch a graph while it runs, step by step, with `stream()`. You can pick what you see
(full state, just the changes, or your own progress messages) and combine several views at once.

## The idea in plain English

`invoke` waits until the whole run is finished and then returns the final state. For a long run that's a long wait
with nothing to look at. **`stream`** runs the same graph but hands you pieces **as they happen**, one after each
step. You loop over them with a `for` loop.

What each piece contains depends on the **stream mode**:

- `"values"`: the **full state** after each step.
- `"updates"`: only **what each node changed** in that step.
- `"custom"`: messages your own nodes send out, such as "multiplying 3 × 4…".
- `"messages"`: an LLM's reply word by word. That needs an LLM, so you'll use it in lesson 19.

**Analogy:** ordering food for delivery. `invoke` is the doorbell when it arrives. `stream` is the tracking page:
"order received", "cooking", "out for delivery". The modes are how much detail the tracking page shows.

## Picture

The lesson 05 calculator. Each arrow is a step, and `stream` gives you one piece after each one. `multiply` also sends
a custom progress message.

```mermaid
flowchart LR
    S([START]) --> parse
    parse -.-> add & subtract & multiply & divide & error
    add & subtract & multiply & divide --> format
    format --> E([END])
    error --> E
    multiply -. "custom: 'multiplying 3 x 4...'" .-> you(["your for loop"])
```

## Step 1: The calculator, with a progress message

This is the lesson 05 graph. The only new thing is in `multiply`. **`get_stream_writer()`** (from `langgraph.config`)
returns a function, `writer`, that sends any value you give it to the `"custom"` stream. Here it sends two messages:
one before the work, one after.

In [1]:
from typing import Literal, TypedDict

from langgraph.config import get_stream_writer
from langgraph.graph import StateGraph, START, END


class CalcState(TypedDict):
    expression: str
    a: float
    op: str
    b: float
    result: float
    answer: str

The nodes. Only `multiply` uses the writer; the others are the same as in lesson 05.

In [2]:
def parse(state: CalcState) -> dict:
    left, op, right = state["expression"].split()
    return {"a": float(left), "op": op, "b": float(right)}


def add(state: CalcState) -> dict:
    return {"result": state["a"] + state["b"]}


def subtract(state: CalcState) -> dict:
    return {"result": state["a"] - state["b"]}


def multiply(state: CalcState) -> dict:
    writer = get_stream_writer()
    writer(f"multiplying {state['a']:g} x {state['b']:g}...")
    result = state["a"] * state["b"]
    writer({"step": "multiply", "done": True})  # any value works, a dict too
    return {"result": result}

The rest of the nodes, and the router, as in lesson 05.

In [3]:
def divide(state: CalcState) -> dict:
    return {"result": state["a"] / state["b"]}


def format_answer(state: CalcState) -> dict:
    return {"answer": f"{state['expression']} = {state['result']:g}"}


def error(state: CalcState) -> dict:
    return {"answer": f"Sorry, I don't know the operator {state['op']!r}."}


OPERATIONS = {"+": "add", "-": "subtract", "*": "multiply", "/": "divide"}


def route(state: CalcState) -> Literal["add", "subtract", "multiply", "divide", "error"]:
    return OPERATIONS.get(state["op"], "error")

The wiring is the lesson 05 graph. Streaming needs no special wiring: only the way you run the graph changes.

In [4]:
builder = StateGraph(CalcState)
builder.add_node("parse", parse)
for name, fn in [("add", add), ("subtract", subtract), ("multiply", multiply), ("divide", divide)]:
    builder.add_node(name, fn)
    builder.add_edge(name, "format")
builder.add_node("format", format_answer)
builder.add_node("error", error)
builder.add_edge(START, "parse")
builder.add_conditional_edges("parse", route)
builder.add_edge("format", END)
builder.add_edge("error", END)

graph = builder.compile()
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse(parse)
	add(add)
	subtract(subtract)
	multiply(multiply)
	divide(divide)
	format(format)
	error(error)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse;
	add --> format;
	divide --> format;
	multiply --> format;
	parse -.-> add;
	parse -.-> divide;
	parse -.-> error;
	parse -.-> multiply;
	parse -.-> subtract;
	subtract --> format;
	error --> __end__;
	format --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 2: `invoke` vs `stream`

`invoke` gives one final answer. `stream` takes the same input but gives back an **iterator**: something you loop over
with `for`, getting one piece at a time as the graph runs.

In [5]:
print("invoke:", graph.invoke({"expression": "3 * 4"}))
print()
for chunk in graph.stream({"expression": "3 * 4"}):
    print("stream:", chunk)

invoke: {'expression': '3 * 4', 'a': 3.0, 'op': '*', 'b': 4.0, 'result': 12.0, 'answer': '3 * 4 = 12'}

stream: {'parse': {'a': 3.0, 'op': '*', 'b': 4.0}}
stream: {'multiply': {'result': 12.0}}
stream: {'format': {'answer': '3 * 4 = 12'}}


With no `stream_mode`, you get `"updates"`, explained in Step 4.

## Step 3: `stream_mode="values"`, the full state after each step

Each piece is the **whole** state at that moment. The first is the input, then one per step. Watch keys appear as
nodes add them.

In [6]:
for state in graph.stream({"expression": "3 * 4"}, stream_mode="values"):
    print(state)

{'expression': '3 * 4'}
{'expression': '3 * 4', 'a': 3.0, 'op': '*', 'b': 4.0}
{'expression': '3 * 4', 'a': 3.0, 'op': '*', 'b': 4.0, 'result': 12.0}
{'expression': '3 * 4', 'a': 3.0, 'op': '*', 'b': 4.0, 'result': 12.0, 'answer': '3 * 4 = 12'}


## Step 4: `stream_mode="updates"`, what each node changed

Each piece is a dict of `{node name: the update it returned}`. This is smaller than `"values"`, and it tells you
**which node** ran, so it's the easiest way to follow the route through a graph.

In [7]:
for update in graph.stream({"expression": "10 / 4"}, stream_mode="updates"):
    for node_name, changes in update.items():
        print(f"{node_name:8} changed {changes}")

parse    changed {'a': 10.0, 'op': '/', 'b': 4.0}
divide   changed {'result': 2.5}
format   changed {'answer': '10 / 4 = 2.5'}


## Step 5: `stream_mode="custom"`, your own messages

Only what nodes send with `writer(...)` comes through. Multiplying sends two messages. Adding sends none, because
`add` never calls the writer.

In [8]:
print("3 * 4:")
for message in graph.stream({"expression": "3 * 4"}, stream_mode="custom"):
    print("  ", message)

print("3 + 4:", list(graph.stream({"expression": "3 + 4"}, stream_mode="custom")))

3 * 4:
   multiplying 3 x 4...
   {'step': 'multiply', 'done': True}
3 + 4: []


## Step 6: Several modes at once

Pass a **list** of modes. Each piece then becomes a pair `(mode, data)`, so you can tell them apart. Here progress
messages and node updates arrive interleaved, in the order they happened.

In [9]:
for mode, data in graph.stream({"expression": "3 * 4"}, stream_mode=["updates", "custom"]):
    print(f"{mode:8} {data}")

updates  {'parse': {'a': 3.0, 'op': '*', 'b': 4.0}}
custom   multiplying 3 x 4...
custom   {'step': 'multiply', 'done': True}
updates  {'multiply': {'result': 12.0}}
updates  {'format': {'answer': '3 * 4 = 12'}}


## Step 7: One shape for every piece, `version="v2"`

With several modes, the pairs work but get fiddly. The LangGraph docs recommend passing `version="v2"`. Every piece is
then a dict with the same three keys: `type` (the mode), `ns` (which graph it came from, used with subgraphs in lesson
16) and `data`.

In [10]:
for part in graph.stream({"expression": "3 * 4"}, stream_mode=["updates", "custom"], version="v2"):
    print(f"type={part['type']:8} ns={part['ns']}  data={part['data']}")

type=updates  ns=()  data={'parse': {'a': 3.0, 'op': '*', 'b': 4.0}}
type=custom   ns=()  data=multiplying 3 x 4...
type=custom   ns=()  data={'step': 'multiply', 'done': True}
type=updates  ns=()  data={'multiply': {'result': 12.0}}
type=updates  ns=()  data={'format': {'answer': '3 * 4 = 12'}}


## Step 8: A look at `stream_events` (newer, experimental)

The installed LangGraph also has `graph.stream_events(..., version="v3")`, a newer event-based stream. The docs point
new applications, such as chat UIs, towards event streaming. In this version it prints a warning that it's still
**experimental**, so this tutorial sticks to `stream()`. Here's one short look at what it gives: each event is a dict
with a `method` (the kind of event) and `params` holding the data.

In [11]:
for event in graph.stream_events({"expression": "3 * 4"}, version="v3"):
    print(event["method"], event["params"]["data"])

values {'expression': '3 * 4'}
values {'expression': '3 * 4', 'a': 3.0, 'op': '*', 'b': 4.0}
values {'expression': '3 * 4', 'a': 3.0, 'op': '*', 'b': 4.0, 'result': 12.0}
values {'expression': '3 * 4', 'a': 3.0, 'op': '*', 'b': 4.0, 'result': 12.0, 'answer': '3 * 4 = 12'}


/home/jupyter/agentic_ai_tutorial/langgraph-calculator-tutorial/.venv/lib/python3.12/site-packages/langgraph/pregel/main.py:3708: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return self._pregel_stream_v3(
/home/jupyter/agentic_ai_tutorial/langgraph-calculator-tutorial/.venv/lib/python3.12/site-packages/langgraph/pregel/main.py:3558: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return GraphRunStream(graph_iter, mux)


The other mode you'll use later is `stream_mode="messages"`. It streams an LLM's answer token by token (a **token** is
a small piece of text, roughly a word). It needs a real LLM, so it's in lesson 19.

## What just happened

- **Step 2** compared the two. `invoke` returned one final dict; `stream` gave three pieces, one per node.
- **Step 3** (`"values"`) printed the full state four times, growing from `{'expression': '3 * 4'}` to the full state
  with `answer: '3 * 4 = 12'`.
- **Step 4** (`"updates"`) printed one line per node: `parse`, `divide`, `format`, showing the route for `10 / 4`.
- **Step 5** (`"custom"`) printed only `multiplying 3 x 4...` and `{'step': 'multiply', 'done': True}`. For `3 + 4`
  it printed nothing: `[]`.
- **Step 6** used two modes. Each piece was a `(mode, data)` pair, and the custom messages appeared between the
  `parse` and `multiply` updates.
- **Step 7** used `version="v2"`, so every piece had the same `type` / `ns` / `data` keys.
- **Step 8** showed `stream_events(version="v3")` with its experimental warning.

## Common mistakes

1. **Calling `get_stream_writer()` outside a running graph.** It only works inside a node while the graph runs:
   `RuntimeError: Called get_config outside of a runnable context`.
   Fix: call it inside the node function, not at the top of the notebook.
2. **Sending custom messages but streaming another mode.** With `stream_mode="updates"` (the default), messages from
   `writer(...)` are **silently dropped**. There's no error.
   Fix: include `"custom"` in `stream_mode`.
3. **Treating pieces as dicts when using a list of modes.** With `stream_mode=["updates", "custom"]` each piece is a
   tuple, so `chunk["parse"]` fails:
   `TypeError: tuple indices must be integers or slices, not str`.
   Fix: unpack it with `for mode, data in ...`, or use `version="v2"` and read `part["type"]` and `part["data"]`.

## Try it

1. Make `divide` send a progress message too, such as `"dividing 10 by 4..."`. Stream `"10 / 4"` with
   `stream_mode="custom"` to check it.
2. Using `stream_mode="updates"`, print **only** the final answer: watch for the piece that comes from the `format`
   node (or the `error` node), and print its `answer`. Try it with `"6 * 7"` and `"2 % 3"`.

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **Streaming / `stream()`**: running a graph and getting results piece by piece while it runs.
- **Stream mode**: what each piece contains: `"values"`, `"updates"`, `"custom"`, `"messages"` and more.
- **`get_stream_writer()`**: gives a node a `writer` function that sends custom data to the `"custom"` stream.
- **Iterator**: something you loop over with `for`, getting one item at a time.
- **Token**: a small piece of text (roughly a word) that an LLM produces one at a time.